In [0]:
%run "./Data transformations and training"

In [0]:
for col in ["lag_1", "lag_7", "rolling_avg_7"]:
    X_train[col] = X_train[col].fillna(0).astype(float)
    X_test[col] = X_test[col].fillna(0).astype(float)

In [0]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6
)

model.fit(X_train, y_train)

In [0]:
predictions = model.predict(X_test)

In [0]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, predictions)

rmse = np.sqrt(mean_squared_error(y_test, predictions))

print("MAE:", mae)
print("RMSE:", rmse)

In [0]:
import shap

explainer = shap.Explainer(model)

shap_values = explainer(X_test)

In [0]:
shap.summary_plot(shap_values, X_test)

In [0]:
shap.plots.waterfall(shap_values[0])

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

plt.plot(y_test.values, label="Actual")
plt.plot(predictions, label="Predicted")

plt.legend()
plt.title("Demand Forecasting")

plt.show()

In [0]:
# ========================================
# XGBoost Algorithm Ended
# ========================================

In [0]:
#################################

In [0]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

rf_model.fit(X_train, y_train)

In [0]:
rf_predictions = rf_model.predict(X_test)

In [0]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))

print("Random Forest MAE:", rf_mae)
print("Random Forest RMSE:", rf_rmse)

In [0]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predictions))

print("Random Forest MAE:", rf_mae)
print("Random Forest RMSE:", rf_rmse)

In [0]:
import shap

rf_explainer = shap.Explainer(rf_model)

rf_shap_values = rf_explainer(X_test)

In [0]:
shap.summary_plot(rf_shap_values, X_test)

In [0]:
shap.summary_plot(rf_shap_values, X_test)

In [0]:
# ========================================
# Random Forest Algorithm Ended
# ========================================

In [0]:
%pip install tensorflow

In [0]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# Reshape data for LSTM: (samples, timesteps, features)
# LSTM expects 3D input: [samples, timesteps, features]
# We'll use timesteps=1 since we have lag features that capture temporal info
X_train_lstm = X_train.values.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_lstm = X_test.values.reshape((X_test.shape[0], 1, X_test.shape[1]))

print(f"LSTM Training data shape: {X_train_lstm.shape}")
print(f"LSTM Test data shape: {X_test_lstm.shape}")

In [0]:
import pandas as pd

# Convert y_train to numeric (LSTM requires numeric dtype)
y_train_numeric = pd.to_numeric(y_train, errors='coerce')

# Build LSTM model
lstm_model = Sequential([
    LSTM(64, activation='relu', input_shape=(1, X_train.shape[1]), return_sequences=True),
    Dropout(0.2),
    LSTM(32, activation='relu'),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

# Compile model
lstm_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

print("LSTM Model Architecture:")
lstm_model.summary()

# Train model
print("\nTraining LSTM model...")
history = lstm_model.fit(
    X_train_lstm, y_train_numeric,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    verbose=0
)

print("\n✓ LSTM model training complete")

In [0]:
# Generate predictions
lstm_predictions = lstm_model.predict(X_test_lstm, verbose=0)
lstm_predictions = lstm_predictions.flatten()  # Convert to 1D array

print(f"Generated {len(lstm_predictions)} LSTM predictions")

In [0]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

lstm_mae = mean_absolute_error(y_test, lstm_predictions)
lstm_rmse = np.sqrt(mean_squared_error(y_test, lstm_predictions))

print("LSTM MAE:", lstm_mae)
print("LSTM RMSE:", lstm_rmse)

In [0]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

# Loss plot
ax1.plot(history.history['loss'], label='Training Loss')
ax1.plot(history.history['val_loss'], label='Validation Loss')
ax1.set_title('LSTM Model Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# MAE plot
ax2.plot(history.history['mae'], label='Training MAE')
ax2.plot(history.history['val_mae'], label='Validation MAE')
ax2.set_title('LSTM Model MAE')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('MAE')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

In [0]:
import shap

# For neural networks, use GradientExplainer (works with TensorFlow/Keras)
# Create a background dataset (sample of training data)
background = X_train_lstm[:100]  # Use 100 samples as background

# Create SHAP explainer for LSTM
lstm_explainer = shap.GradientExplainer(lstm_model, background)

# Calculate SHAP values for test set (use subset for speed)
test_sample = X_test_lstm[:100]
lstm_shap_values = lstm_explainer.shap_values(test_sample)

print(f"✓ Generated SHAP values for {len(test_sample)} test samples")
print(f"SHAP values shape: {np.array(lstm_shap_values).shape}")

In [0]:
import numpy as np
import pandas as pd

# Reshape SHAP values and test data for visualization
# Remove the timestep dimension for plotting
lstm_shap_2d = np.array(lstm_shap_values).squeeze()  # Shape: (100, 6)
test_sample_2d = test_sample.squeeze()  # Shape: (100, 6)

# Create a DataFrame for better feature names in the plot
test_sample_df = pd.DataFrame(test_sample_2d, columns=X_train.columns)

# Generate SHAP summary plot
shap.summary_plot(lstm_shap_2d, test_sample_df)

In [0]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

plt.plot(y_test.values, label="Actual", linewidth=2, color='black')
plt.plot(predictions, label="XGBoost", alpha=0.7)
plt.plot(rf_predictions, label="Random Forest", alpha=0.7)
plt.plot(lstm_predictions, label="LSTM", alpha=0.7)

plt.legend(loc='best')
plt.title("Demand Forecasting: All Models Comparison", fontsize=14)
plt.xlabel("Time")
plt.ylabel("Quantity")
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [0]:
# ========================================
# LSTM (RNN) Algorithm Ended
# ========================================

In [0]:
import pandas as pd

# Create comparison table with all three models
comparison = pd.DataFrame({
    "Model": ["XGBoost", "Random Forest", "LSTM"],
    "MAE": [mae, rf_mae, lstm_mae],
    "RMSE": [rmse, rf_rmse, lstm_rmse]
})

# Sort by MAE (lower is better)
comparison = comparison.sort_values('MAE').reset_index(drop=True)

print("\n" + "="*60)
print("MODEL PERFORMANCE COMPARISON")
print("="*60)
display(comparison)
print("="*60)

# Highlight best model
best_model = comparison.iloc[0]['Model']
best_mae = comparison.iloc[0]['MAE']
best_rmse = comparison.iloc[0]['RMSE']

print(f"\n✓ Best performing model: {best_model}")
print(f"  MAE:  {best_mae:.2f}")
print(f"  RMSE: {best_rmse:.2f}")

In [0]:
results = pd.DataFrame({
    "Actual": y_test.values,
    "XGBoost_Prediction": predictions,
    "RandomForest_Prediction": rf_predictions,
    "LSTM_Prediction": lstm_predictions
})

results.head()

In [0]:
spark_results = spark.createDataFrame(results)

display(spark_results)

In [0]:
spark.sql('DROP TABLE IF EXISTS workspace.general_use.demand_forecast_results')
spark_results.write.mode("overwrite").saveAsTable("workspace.general_use.demand_forecast_results")

In [0]:
spark_results.write.mode("overwrite").saveAsTable(
    "workspace.general_use.demand_forecast_results"
)

In [0]:
%sql
use catalog `workspace`; select * from `general_use`.`demand_forecast_results` limit 100;